In [ ]:
# ============================================
# STEP 1: IMPORT LIBRARIES
# ============================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score

from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import dendrogram, linkage

import warnings
warnings.filterwarnings("ignore")

print("Libraries imported successfully!")

In [ ]:
# ============================================
# STEP 2: LOAD DATA
# ============================================

clients = pd.read_csv('/content/clients.csv')
properties = pd.read_csv('/content/properties.csv')

print("Clients dataset shape:", clients.shape)
print("Properties dataset shape:", properties.shape)

In [ ]:
clients.head()

In [ ]:
clients.info()

In [ ]:
clients.describe(include='all').T

In [ ]:
clients.isnull().sum()

In [ ]:
properties.head()

In [ ]:
properties.info()

In [ ]:
properties.isnull().sum()

In [ ]:
properties['listing_status'].value_counts()

In [ ]:
properties['sale_price'] = (
    properties['sale_price']
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .astype(float)
)

In [ ]:
# Original client date field was converted to datetime during preprocessing.
# The public repository does not include the original client-level date field.


In [ ]:
properties['transaction_date'] = pd.to_datetime(
    properties['transaction_date'],
    errors='coerce'
)

In [ ]:
# Original client date field was inspected during private preprocessing.
# Personal date values are intentionally excluded from the public notebook.


In [ ]:
properties['transaction_date'].head()

In [ ]:
# Age was derived during the private preprocessing stage.
# The resulting age feature is retained in the public analysis.


In [ ]:
# Age is used as a clustering feature.
# Original client-level date values are not included in the public project.


In [ ]:
clients['age'].describe()

In [ ]:
sold_properties = properties[
    properties['listing_status'] == 'Sold'
].copy()

print("Sold properties:", sold_properties.shape)

In [ ]:
# Property transactions were aggregated by the internal client reference
# during private preprocessing.
#
# The public repository contains only the resulting buyer-level aggregates.


In [ ]:
buyer_property_summary.head()

In [ ]:
# Buyer-level client and property data were merged during private preprocessing.
# The public repository contains the resulting sanitized buyer-level dataset.


In [ ]:
buyer_data.shape

In [ ]:
buyer_data.head()

In [ ]:
transaction_features = [
    'total_properties',
    'total_spend',
    'average_property_price',
    'maximum_property_price',
    'total_area_sqft',
    'average_area_sqft'
]

buyer_data[transaction_features] = (
    buyer_data[transaction_features].fillna(0)
)

In [ ]:
buyer_data.isnull().sum()

In [ ]:
buyer_data['is_investor'] = (
    buyer_data['acquisition_purpose']
    .str.lower()
    .eq('investment')
    .astype(int)
)

In [ ]:
buyer_data['loan_flag'] = (
    buyer_data['loan_applied']
    .str.lower()
    .eq('yes')
    .astype(int)
)

In [ ]:
buyer_data['company_flag'] = (
    buyer_data['client_type']
    .str.lower()
    .eq('company')
    .astype(int)
)

In [ ]:
buyer_data['avg_spend_per_property'] = np.where(
    buyer_data['total_properties'] > 0,
    buyer_data['total_spend'] / buyer_data['total_properties'],
    0
)

In [ ]:
plt.figure(figsize=(8,5))

sns.countplot(
    data=buyer_data,
    x='acquisition_purpose'
)

plt.title('Buyer Acquisition Purpose')
plt.xlabel('Acquisition Purpose')
plt.ylabel('Number of Buyers')

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

sns.countplot(
    data=buyer_data,
    x='client_type'
)

plt.title('Buyer Type Distribution')
plt.xlabel('Client Type')
plt.ylabel('Number of Buyers')

plt.show()

In [ ]:
investment_summary = (
    buyer_data
    .groupby('acquisition_purpose')
    .agg(
        buyers=('client_id', 'count'),
        average_spend=('total_spend', 'mean'),
        average_properties=('total_properties', 'mean'),
        average_property_price=('average_property_price', 'mean')
    )
    .reset_index()
)

investment_summary

In [ ]:
country_summary = (
    buyer_data
    .groupby('country')
    .agg(
        buyers=('client_id', 'count'),
        average_spend=('total_spend', 'mean'),
        investment_buyers=('is_investor', 'sum')
    )
    .sort_values('buyers', ascending=False)
)

country_summary

In [ ]:
plt.figure(figsize=(12,6))

sns.countplot(
    data=buyer_data,
    y='country',
    order=buyer_data['country'].value_counts().index
)

plt.title('Buyers by Country')
plt.xlabel('Number of Buyers')
plt.ylabel('Country')

plt.show()

In [ ]:
numeric_features = [
    'age',
    'satisfaction_score',
    'total_properties',
    'total_spend',
    'average_property_price',
    'maximum_property_price',
    'total_area_sqft',
    'average_area_sqft',
    'avg_spend_per_property'
]

categorical_features = [
    'client_type',
    'gender',
    'country',
    'region',
    'acquisition_purpose',
    'loan_applied',
    'referral_channel'
]

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            'num',
            StandardScaler(),
            numeric_features
        ),
        (
            'cat',
            OneHotEncoder(
                handle_unknown='ignore',
                sparse_output=False
            ),
            categorical_features
        )
    ]
)

In [ ]:
X = buyer_data[
    numeric_features + categorical_features
]

In [ ]:
X_processed = preprocessor.fit_transform(X)

print("Processed data shape:", X_processed.shape)

In [ ]:
# ============================================
# CHECK MISSING VALUES BEFORE CLUSTERING
# ============================================

print("Missing values in buyer_data:")
print(
    buyer_data[
        numeric_features + categorical_features
    ].isnull().sum().sort_values(ascending=False)
)

In [ ]:
print("Total missing values:",
      buyer_data[numeric_features + categorical_features].isnull().sum().sum())

In [ ]:
print("Missing ages:", buyer_data['age'].isnull().sum())
print("Total buyers:", len(buyer_data))

In [ ]:
# Missing-age validation was performed during private preprocessing.
# The public notebook does not display original client-level date values.


In [ ]:
buyer_data[numeric_features].isnull().sum()

In [ ]:
buyer_data[numeric_features].describe().T

In [ ]:
# ============================================
# HANDLE MISSING NUMERIC VALUES
# ============================================

for column in numeric_features:
    buyer_data[column] = buyer_data[column].fillna(
        buyer_data[column].median()
    )

In [ ]:
buyer_data[numeric_features].isnull().sum()

In [ ]:
# ============================================
# HANDLE MISSING CATEGORICAL VALUES
# ============================================

for column in categorical_features:
    buyer_data[column] = buyer_data[column].fillna('Unknown')

In [ ]:
buyer_data[categorical_features].isnull().sum()

In [ ]:
clustering_data = buyer_data[
    numeric_features + categorical_features
].copy()

print("Total missing values:")
print(clustering_data.isnull().sum().sum())

In [ ]:
# ============================================
# PREPROCESSING
# ============================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            'num',
            StandardScaler(),
            numeric_features
        ),
        (
            'cat',
            OneHotEncoder(
                handle_unknown='ignore',
                sparse_output=False
            ),
            categorical_features
        )
    ]
)

X = buyer_data[
    numeric_features + categorical_features
].copy()

X_processed = preprocessor.fit_transform(X)

print("Processed data shape:", X_processed.shape)

In [ ]:
print("NaN values:", np.isnan(X_processed).sum())

In [ ]:
print("Infinite values:", np.isinf(X_processed).sum())

In [ ]:
# ============================================
# STEP 13: FIND OPTIMAL NUMBER OF CLUSTERS
# ============================================

inertia = []
silhouette_scores = []

K_range = range(2, 9)

for k in K_range:

    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = kmeans.fit_predict(X_processed)

    inertia.append(kmeans.inertia_)

    silhouette_scores.append(
        silhouette_score(
            X_processed,
            labels
        )

    )

print("Analysis completed successfully.")

In [ ]:
# ============================================
# ELBOW METHOD
# ============================================

plt.figure(figsize=(8, 5))

plt.plot(
    list(K_range),
    inertia,
    marker='o'
)

plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.title('Elbow Method for Optimal K')

plt.xticks(list(K_range))

plt.show()

In [ ]:
# ============================================
# SILHOUETTE SCORE
# ============================================

plt.figure(figsize=(8, 5))

plt.plot(
    list(K_range),
    silhouette_scores,
    marker='o'
)

plt.xlabel('Number of Clusters (K)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score by Number of Clusters')

plt.xticks(list(K_range))

plt.show()

In [ ]:
# ============================================
# DISPLAY CLUSTER EVALUATION RESULTS
# ============================================

cluster_evaluation = pd.DataFrame({
    'K': list(K_range),
    'Inertia': inertia,
    'Silhouette Score': silhouette_scores
})

cluster_evaluation

In [ ]:
best_k_by_silhouette = (
    cluster_evaluation
    .loc[
        cluster_evaluation['Silhouette Score'].idxmax(),
        'K'
    ]
)

best_score = (
    cluster_evaluation['Silhouette Score'].max()
)

print("Best K according to Silhouette Score:",
      best_k_by_silhouette)

print("Best Silhouette Score:",
      round(best_score, 4))

In [ ]:
print("Original feature count:", len(numeric_features + categorical_features))
print("Processed feature count:", X_processed.shape[1])

In [ ]:
cluster_evaluation

In [ ]:
print(
    "Best K according to Silhouette Score:",
    best_k_by_silhouette
)

print(
    "Best Silhouette Score:",
    round(best_score, 4)
)

In [ ]:
# ============================================
# FINAL K-MEANS MODEL
# ============================================

optimal_k = 3

kmeans = KMeans(
    n_clusters=optimal_k,
    random_state=42,
    n_init=10
)

buyer_data['cluster'] = kmeans.fit_predict(X_processed)

print("K-Means model created successfully.")
print()
print("Number of clusters:", optimal_k)

In [ ]:
# ============================================
# CLUSTER SIZE
# ============================================

cluster_counts = (
    buyer_data['cluster']
    .value_counts()
    .sort_index()
)

print(cluster_counts)

In [ ]:
final_silhouette = silhouette_score(
    X_processed,
    buyer_data['cluster']
)

print(
    "Final K-Means Silhouette Score:",
    round(final_silhouette, 4)
)

In [ ]:
# ============================================
# CLUSTER PROFILE
# ============================================

cluster_profile = (
    buyer_data
    .groupby('cluster')
    .agg(
        buyer_count=('client_id', 'count'),
        avg_age=('age', 'mean'),
        avg_satisfaction=('satisfaction_score', 'mean'),
        investment_rate=('is_investor', 'mean'),
        loan_rate=('loan_flag', 'mean'),
        company_rate=('company_flag', 'mean'),
        avg_properties=('total_properties', 'mean'),
        avg_total_spend=('total_spend', 'mean'),
        avg_property_price=('average_property_price', 'mean'),
        avg_area=('average_area_sqft', 'mean')
    )
    .round(2)
)

cluster_profile

In [ ]:
cluster_profile_display = cluster_profile.copy()

cluster_profile_display['investment_rate'] = (
    cluster_profile_display['investment_rate'] * 100
).round(2)

cluster_profile_display['loan_rate'] = (
    cluster_profile_display['loan_rate'] * 100
).round(2)

cluster_profile_display['company_rate'] = (
    cluster_profile_display['company_rate'] * 100
).round(2)

cluster_profile_display

In [ ]:
purpose_by_cluster = pd.crosstab(
    buyer_data['cluster'],
    buyer_data['acquisition_purpose'],
    normalize='index'
) * 100

purpose_by_cluster.round(2)

In [ ]:
loan_by_cluster = pd.crosstab(
    buyer_data['cluster'],
    buyer_data['loan_applied'],
    normalize='index'
) * 100

loan_by_cluster.round(2)

In [ ]:
client_type_by_cluster = pd.crosstab(
    buyer_data['cluster'],
    buyer_data['client_type'],
    normalize='index'
) * 100

client_type_by_cluster.round(2)

In [ ]:
country_by_cluster = pd.crosstab(
    buyer_data['cluster'],
    buyer_data['country'],
    normalize='index'
) * 100

country_by_cluster.round(2)

In [ ]:
region_by_cluster = pd.crosstab(
    buyer_data['cluster'],
    buyer_data['region'],
    normalize='index'
) * 100

region_by_cluster.round(2)

In [ ]:
plt.figure(figsize=(9, 5))

sns.barplot(
    data=buyer_data,
    x='cluster',
    y='total_spend',
    estimator='mean'
)

plt.title('Average Total Property Spend by Buyer Cluster')
plt.xlabel('Cluster')
plt.ylabel('Average Total Spend')

plt.show()

In [ ]:
plt.figure(figsize=(9, 5))

sns.barplot(
    data=buyer_data,
    x='cluster',
    y='age',
    estimator='mean'
)

plt.title('Average Buyer Age by Cluster')
plt.xlabel('Cluster')
plt.ylabel('Average Age')

plt.show()

In [ ]:
# ============================================
# PCA VISUALIZATION
# ============================================

pca = PCA(
    n_components=2,
    random_state=42
)

X_pca = pca.fit_transform(X_processed)

pca_df = pd.DataFrame(
    X_pca,
    columns=['PC1', 'PC2']
)

pca_df['cluster'] = buyer_data['cluster'].values

In [ ]:
plt.figure(figsize=(10, 7))

sns.scatterplot(
    data=pca_df,
    x='PC1',
    y='PC2',
    hue='cluster',
    palette='tab10'
)

plt.title('Buyer Segments - PCA Visualization')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')

plt.show()

In [ ]:
# ============================================
# HIERARCHICAL CLUSTERING SAMPLE
# ============================================

sample_size = min(500, len(X_processed))

random_state = np.random.RandomState(42)

sample_indices = random_state.choice(
    len(X_processed),
    size=sample_size,
    replace=False
)

X_sample = X_processed[sample_indices]

print("Hierarchical clustering sample size:",
      X_sample.shape)

In [ ]:
linkage_matrix = linkage(
    X_sample,
    method='ward'
)

print("Linkage matrix created successfully.")

In [ ]:
plt.figure(figsize=(14, 7))

dendrogram(
    linkage_matrix,
    truncate_mode='lastp',
    p=30
)

plt.title(
    'Hierarchical Clustering Dendrogram'
)

plt.xlabel('Buyer Groups')
plt.ylabel('Distance')

plt.show()

In [ ]:
hierarchical = AgglomerativeClustering(
    n_clusters=3,
    linkage='ward'
)

hierarchical_labels = hierarchical.fit_predict(
    X_processed
)

hierarchical_silhouette = silhouette_score(
    X_processed,
    hierarchical_labels
)

print(
    "K-Means Silhouette:",
    round(final_silhouette, 4)
)

print(
    "Hierarchical Silhouette:",
    round(hierarchical_silhouette, 4)
)

In [ ]:
cluster_profile_display

In [ ]:
purpose_by_cluster.round(2)

In [ ]:
loan_by_cluster.round(2)

In [ ]:
client_type_by_cluster.round(2)

In [ ]:
cluster_distributioncluster_counts = buyer_data['cluster'].value_counts().sort_index()

cluster_distribution = pd.DataFrame({
    'Buyer Count': cluster_counts,
    'Percentage': (cluster_counts / len(buyer_data) * 100).round(2)
})

display(cluster_distribution)

In [ ]:
cluster_profile = (
    buyer_data
    .groupby('cluster')
    .agg(
        buyer_count=('client_id', 'count'),
        avg_age=('age', 'mean'),
        avg_satisfaction=('satisfaction_score', 'mean'),
        investment_rate=('is_investor', 'mean'),
        loan_rate=('loan_flag', 'mean'),
        company_rate=('company_flag', 'mean'),
        avg_properties=('total_properties', 'mean'),
        avg_total_spend=('total_spend', 'mean'),
        avg_property_price=('average_property_price', 'mean'),
        avg_area=('average_area_sqft', 'mean')
    )
    .round(2)
)

cluster_profile_display = cluster_profile.copy()

cluster_profile_display['investment_rate'] = (
    cluster_profile_display['investment_rate'] * 100
).round(2)

cluster_profile_display['loan_rate'] = (
    cluster_profile_display['loan_rate'] * 100
).round(2)

cluster_profile_display['company_rate'] = (
    cluster_profile_display['company_rate'] * 100
).round(2)

display(cluster_profile_display)

In [ ]:
purpose_by_cluster = pd.crosstab(
    buyer_data['cluster'],
    buyer_data['acquisition_purpose'],
    normalize='index'
) * 100

display(purpose_by_cluster.round(2))

In [ ]:
client_type_by_cluster = pd.crosstab(
    buyer_data['cluster'],
    buyer_data['client_type'],
    normalize='index'
) * 100

display(client_type_by_cluster.round(2))

In [ ]:
loan_by_cluster = pd.crosstab(
    buyer_data['cluster'],
    buyer_data['loan_applied'],
    normalize='index'
) * 100

display(loan_by_cluster.round(2))

In [ ]:
print(buyer_data['loan_applied'].value_counts(dropna=False))

In [ ]:
cluster_profile = (
    buyer_data
    .groupby('cluster')
    .agg(
        buyer_count=('client_id', 'count'),
        avg_age=('age', 'mean'),
        avg_satisfaction=('satisfaction_score', 'mean'),
        investment_rate=('is_investor', 'mean'),
        loan_rate=('loan_flag', 'mean'),
        company_rate=('company_flag', 'mean'),
        avg_properties=('total_properties', 'mean'),
        avg_total_spend=('total_spend', 'mean'),
        avg_property_price=('average_property_price', 'mean'),
        avg_area=('average_area_sqft', 'mean')
    )
)

cluster_profile_display = cluster_profile.copy()

# Convert rates to percentages
cluster_profile_display['investment_rate'] = (
    cluster_profile_display['investment_rate'] * 100
).round(2)

cluster_profile_display['loan_rate'] = (
    cluster_profile_display['loan_rate'] * 100
).round(2)

cluster_profile_display['company_rate'] = (
    cluster_profile_display['company_rate'] * 100
).round(2)

# Round remaining numeric columns
cluster_profile_display = cluster_profile_display.round(2)

display(cluster_profile_display)

In [ ]:
purpose_by_cluster = pd.crosstab(
    buyer_data['cluster'],
    buyer_data['acquisition_purpose'],
    normalize='index'
) * 100

display(purpose_by_cluster.round(2))

In [ ]:
client_type_by_cluster = pd.crosstab(
    buyer_data['cluster'],
    buyer_data['client_type'],
    normalize='index'
) * 100

display(client_type_by_cluster.round(2))

In [ ]:
country_by_cluster = pd.crosstab(
    buyer_data['cluster'],
    buyer_data['country'],
    normalize='index'
) * 100

display(country_by_cluster.round(2))

In [ ]:
region_by_cluster = pd.crosstab(
    buyer_data['cluster'],
    buyer_data['region'],
    normalize='index'
) * 100

display(region_by_cluster.round(2))

In [ ]:
import matplotlib.pyplot as plt

cluster_profile_display[
    [
        'avg_properties',
        'avg_total_spend',
        'avg_property_price',
        'avg_area'
    ]
].plot(
    kind='bar',
    figsize=(12, 6)
)

plt.title('Buyer Cluster Comparison')
plt.xlabel('Cluster')
plt.ylabel('Value')
plt.xticks(rotation=0)
plt.legend(
    ['Average Properties',
     'Average Total Spend',
     'Average Property Price',
     'Average Area'],
    bbox_to_anchor=(1.05, 1),
    loc='upper left'
)
plt.tight_layout()
plt.show()

In [ ]:
cluster_names = {
    0: 'High-Activity Investment-Oriented Buyers',
    1: 'Mainstream Lower-Value Buyers',
    2: 'Higher-Value Property Buyers'
}

buyer_data['segment'] = buyer_data['cluster'].map(cluster_names)

buyer_data[['client_id', 'cluster', 'segment']].head(10)

In [ ]:
segment_summary = (
    buyer_data
    .groupby(['cluster', 'segment'])
    .agg(
        buyer_count=('client_id', 'count'),
        avg_age=('age', 'mean'),
        avg_properties=('total_properties', 'mean'),
        avg_total_spend=('total_spend', 'mean'),
        avg_property_price=('average_property_price', 'mean'),
        avg_area=('average_area_sqft', 'mean'),
        investment_rate=('is_investor', 'mean'),
        loan_rate=('loan_flag', 'mean'),
        avg_satisfaction=('satisfaction_score', 'mean')
    )
    .reset_index()
)

segment_summary['buyer_percentage'] = (
    segment_summary['buyer_count'] /
    len(buyer_data) * 100
).round(2)

segment_summary['investment_rate'] = (
    segment_summary['investment_rate'] * 100
).round(2)

segment_summary['loan_rate'] = (
    segment_summary['loan_rate'] * 100
).round(2)

segment_summary = segment_summary.round(2)

display(segment_summary)

In [ ]:
final_segments = buyer_data[
    [
        'client_id',
        'client_type',
        'gender',
        'country',
        'region',
        'age',
        'acquisition_purpose',
        'loan_applied',
        'referral_channel',
        'satisfaction_score',
        'total_properties',
        'total_spend',
        'average_property_price',
        'maximum_property_price',
        'total_area_sqft',
        'average_area_sqft',
        'avg_spend_per_property',
        'cluster',
        'segment'
    ]
].copy()

display(final_segments.head())

In [ ]:
print("Rows:", len(final_segments))
print("Columns:", len(final_segments.columns))
print("Missing values:", final_segments.isna().sum().sum())

In [ ]:
final_segments.to_csv(
    'buyer_segments.csv',
    index=False
)

segment_summary.to_csv(
    'segment_profile.csv',
    index=False
)

print("Files created successfully!")

In [ ]:
cluster_names = {
    0: 'High-Activity Investment-Oriented Buyers',
    1: 'Mainstream Lower-Value Buyers',
    2: 'Higher-Value Property Buyers'
}

buyer_data['segment'] = buyer_data['cluster'].map(cluster_names)

In [ ]:
final_segments = buyer_data[
    [
        'client_id',
        'client_type',
        'gender',
        'country',
        'region',
        'age',
        'acquisition_purpose',
        'loan_applied',
        'referral_channel',
        'satisfaction_score',
        'total_properties',
        'total_spend',
        'average_property_price',
        'maximum_property_price',
        'total_area_sqft',
        'average_area_sqft',
        'avg_spend_per_property',
        'cluster',
        'segment'
    ]
].copy()

print("Rows:", len(final_segments))
print("Columns:", len(final_segments.columns))
print("Missing values:", final_segments.isna().sum().sum())

display(final_segments.head())

In [ ]:
segment_counts = (
    final_segments['segment']
    .value_counts()
    .reset_index()
)

segment_counts.columns = ['Segment', 'Buyer Count']

segment_counts['Percentage'] = (
    segment_counts['Buyer Count'] /
    segment_counts['Buyer Count'].sum() * 100
).round(2)

display(segment_counts)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

plt.bar(
    segment_counts['Segment'],
    segment_counts['Buyer Count']
)

plt.title('Buyer Distribution by Segment')
plt.xlabel('Buyer Segment')
plt.ylabel('Number of Buyers')
plt.xticks(rotation=20, ha='right')

plt.tight_layout()
plt.show()

In [ ]:
spending_comparison = (
    final_segments
    .groupby('segment')['total_spend']
    .mean()
    .sort_values(ascending=False)
)

display(spending_comparison)

In [ ]:
plt.figure(figsize=(10, 6))

plt.bar(
    spending_comparison.index,
    spending_comparison.values
)

plt.title('Average Total Spending by Buyer Segment')
plt.xlabel('Buyer Segment')
plt.ylabel('Average Total Spend')

plt.xticks(rotation=20, ha='right')

plt.tight_layout()
plt.show()

In [ ]:
property_comparison = (
    final_segments
    .groupby('segment')['total_properties']
    .mean()
    .sort_values(ascending=False)
)

display(property_comparison)

In [ ]:
plt.figure(figsize=(10, 6))

plt.bar(
    property_comparison.index,
    property_comparison.values
)

plt.title('Average Number of Properties per Buyer')
plt.xlabel('Buyer Segment')
plt.ylabel('Average Properties')

plt.xticks(rotation=20, ha='right')

plt.tight_layout()
plt.show()

In [ ]:
price_comparison = (
    final_segments
    .groupby('segment')['average_property_price']
    .mean()
    .sort_values(ascending=False)
)

display(price_comparison)

In [ ]:
plt.figure(figsize=(10, 6))

plt.bar(
    price_comparison.index,
    price_comparison.values
)

plt.title('Average Property Price by Buyer Segment')
plt.xlabel('Buyer Segment')
plt.ylabel('Average Property Price')

plt.xticks(rotation=20, ha='right')

plt.tight_layout()
plt.show()

In [ ]:
# Investment vs Home-use behavior by segment

investment_comparison = (
    pd.crosstab(
        final_segments['segment'],
        final_segments['acquisition_purpose'],
        normalize='index'
    ) * 100
)

investment_comparison = investment_comparison.round(2)

display(investment_comparison)

In [ ]:
investment_comparison.plot(
    kind='bar',
    figsize=(10,6)
)

plt.title('Acquisition Purpose by Buyer Segment')
plt.xlabel('Buyer Segment')
plt.ylabel('Percentage of Buyers')
plt.xticks(rotation=20, ha='right')
plt.legend(title='Acquisition Purpose')

plt.tight_layout()
plt.show()

In [ ]:
# Buyer segments by country

country_segment = (
    pd.crosstab(
        final_segments['country'],
        final_segments['segment']
    )
)

display(country_segment)

In [ ]:
country_totals = (
    final_segments['country']
    .value_counts()
    .head(10)
)

display(country_totals)

In [ ]:
plt.figure(figsize=(10,6))

plt.bar(
    country_totals.index,
    country_totals.values
)

plt.title('Top 10 Countries by Number of Buyers')
plt.xlabel('Country')
plt.ylabel('Number of Buyers')

plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.show()

In [ ]:
region_totals = (
    final_segments['region']
    .value_counts()
    .head(15)
)

display(region_totals)

In [ ]:
plt.figure(figsize=(12,7))

plt.bar(
    region_totals.index,
    region_totals.values
)

plt.title('Top 15 Regions by Number of Buyers')
plt.xlabel('Region')
plt.ylabel('Number of Buyers')

plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.show()

In [ ]:
top_regions = region_totals.index

region_segment = (
    pd.crosstab(
        final_segments[final_segments['region'].isin(top_regions)]['region'],
        final_segments[final_segments['region'].isin(top_regions)]['segment']
    )
)

display(region_segment)

In [ ]:
region_segment_percentage = (
    pd.crosstab(
        final_segments['region'],
        final_segments['segment'],
        normalize='index'
    ) * 100
)

region_segment_percentage = region_segment_percentage.round(2)

display(region_segment_percentage.loc[top_regions])

In [ ]:
region_segment_percentage.loc[top_regions].plot(
    kind='bar',
    stacked=True,
    figsize=(12,7)
)

plt.title('Buyer Segment Composition by Region')
plt.xlabel('Region')
plt.ylabel('Percentage of Buyers')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Buyer Segment')

plt.tight_layout()
plt.show()

In [ ]:
segment_summary = (
    buyer_data.groupby(['cluster', 'segment'])
    .agg(
        buyer_count=('client_id', 'count'),
        avg_age=('age', 'mean'),
        avg_properties=('total_properties', 'mean'),
        avg_total_spend=('total_spend', 'mean'),
        avg_property_price=('average_property_price', 'mean'),
        avg_area=('average_area_sqft', 'mean'),
        investment_rate=('is_investor', 'mean'),
        loan_rate=('loan_flag', 'mean'),
        avg_satisfaction=('satisfaction_score', 'mean')
    )
    .reset_index()
)

# Convert rates to percentages
segment_summary['buyer_percentage'] = (
    segment_summary['buyer_count'] /
    len(buyer_data) * 100
)

segment_summary['investment_rate'] = (
    segment_summary['investment_rate'] * 100
)

segment_summary['loan_rate'] = (
    segment_summary['loan_rate'] * 100
)

# Round numerical values
segment_summary = segment_summary.round(2)

display(segment_summary)

In [ ]:
final_segments.to_csv('buyer_segments.csv', index=False)

segment_summary.to_csv('segment_profile.csv', index=False)

print("Files saved successfully.")

In [ ]:
# Save final buyer-level segmentation data
final_segments.to_csv('buyer_segments.csv', index=False)

# Save final segment profile
segment_summary.to_csv('segment_profile.csv', index=False)

print("Files saved successfully.")

In [ ]:
import os

print("buyer_segments.csv exists:", os.path.exists('buyer_segments.csv'))
print("segment_profile.csv exists:", os.path.exists('segment_profile.csv'))

**Create the Streamlit application file**

In [ ]:
!pip install -q streamlit

In [ ]:
%%writefile app.py

import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt

# Page configuration
st.set_page_config(
    page_title="Real Estate Buyer Segmentation",
    page_icon="🏠",
    layout="wide"
)

# Title
st.title("🏠 Real Estate Buyer Segmentation & Investment Profiling")
st.write(
    "Interactive dashboard for analyzing buyer segments, "
    "investment behavior, spending patterns, and geographic distribution."
)

# Load data
buyer_data = pd.read_csv("buyer_segments.csv")
segment_profile = pd.read_csv("segment_profile.csv")

st.success("Data loaded successfully!")

# Basic information
st.subheader("Dataset Overview")

col1, col2, col3 = st.columns(3)

with col1:
    st.metric(
        "Total Buyers",
        len(buyer_data)
    )

with col2:
    st.metric(
        "Buyer Segments",
        buyer_data["segment"].nunique()
    )

with col3:
    st.metric(
        "Countries",
        buyer_data["country"].nunique()
    )

# Show segment profile
st.subheader("Buyer Segment Profile")

st.dataframe(
    segment_profile,
    use_container_width=True
)

In [ ]:
import os

print("app.py exists:", os.path.exists("app.py"))

In [ ]:
import os

print("Files currently in Colab:")
for file in os.listdir():
    print(file)

In [ ]:
import os

print(os.path.exists("buyer_segmentation.ipynb"))

In [ ]:
import os

print("IPYNB files found in Colab:")

for file in os.listdir():
    if file.endswith(".ipynb"):
        print(file)